In [1]:
!pip install torchreid gradio tabulate gdown

import os
import pickle
import torch
import numpy as np
import pandas as pd
import gdown
from PIL import Image
from torchvision import transforms
import torchreid
from torchreid import utils
import gradio as gr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=a2f1cc351efd1db2daa496a146328d7b56e8b56c869c1a8971dbc7cca935b427
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid


/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
2026-07-08 04:13:28.397742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783484008.792152      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783484008.918498      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783484009.916332      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783484009.916380      58 computation_placer.cc:17

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATASET_ROOT    = "/kaggle/input/datasets/singh96divya/wb-wob-reid-dataset/WB_WoB-ReID"
PKL_ROOT        = "/kaggle/input/datasets/reymonthatarigan/my-model-files"
SUBSET_LIST     = ["with_bag", "without_bag", "both_small", "both_large"]
MODEL_NAME_LIST = ["Pretrained ImageNet", "Pretrained Market-1501", "Pretrained MSMT17"]
TOP_K           = 10

MODEL_CONFIGS = [
    {
        "name"       : "Pretrained ImageNet",
        "num_classes": 1000,
        "weight_url" : None,
        "weight_file": None,
    },
    {
        "name"       : "Pretrained Market-1501",
        "num_classes": 751,
        "weight_url" : "https://drive.google.com/uc?id=1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA",
        "weight_file": "osnet_x1_0_market1501.pth",
    },
    {
        "name"       : "Pretrained MSMT17",
        "num_classes": 1041,
        "weight_url" : "https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M",
        "weight_file": "osnet_x1_0_msmt17.pth",
    },
]

transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Device: cuda


In [3]:
def load_images_from_folder(folder_path):
    """Load semua gambar .jpg dari folder, parse person_id dan camera_id dari nama file."""
    data = []
    for fname in sorted(os.listdir(folder_path)):
        if not fname.endswith('.jpg'):
            continue
        parts = fname.split('_')
        try:
            person_id = int(parts[0])
            camera_id = int(parts[1][1:])
        except:
            continue
        if person_id == -1:
            continue
        data.append((os.path.join(folder_path, fname), person_id, camera_id))
    return data


def extract_features_from_paths(data_list, model, transform, device, batch_size=32):
    """Ekstrak fitur dari list (path, pid, cid) menggunakan model."""
    features_list, labels_list, cameras_list = [], [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(data_list), batch_size):
            batch = data_list[i:i + batch_size]
            imgs  = torch.stack([
                transform(Image.open(p).convert('RGB')) for p, _, _ in batch
            ]).to(device)
            features_list.append(model(imgs).cpu().numpy())
            labels_list.extend([x[1] for x in batch])
            cameras_list.extend([x[2] for x in batch])
            if i % (batch_size * 10) == 0:
                print(f"  Processed {i}/{len(data_list)}")
    return np.vstack(features_list), labels_list, cameras_list


def compute_cosine_distance(qf, gf):
    """Hitung cosine distance antara query features dan gallery features."""
    q = qf / np.linalg.norm(qf, axis=1, keepdims=True)
    g = gf / np.linalg.norm(gf, axis=1, keepdims=True)
    return 1 - np.dot(q, g.T)


def parse_person_id(filepath):
    """Parse person ID dari nama file format XXXX_cY_fZZZZ.jpg. Return None jika gagal."""
    if filepath is None:
        return None
    try:
        return int(os.path.basename(filepath).split("_")[0])
    except:
        return None


def make_retrieval_outputs(indices, dist_arr, g_paths, g_labels, query_pid):
    """Buat gallery images dan tabel markdown dari hasil retrieval."""
    images, rows = [], []
    for rank, idx in enumerate(indices):
        pid      = g_labels[idx]
        score    = dist_arr[idx]
        is_match = "✅ True" if pid == query_pid else "❌ False"
        images.append((Image.open(g_paths[idx]).convert("RGB"),
                       f"Rank {rank+1} | ID:{pid} | dist:{score:.3f}"))
        rows.append({"Rank": rank + 1, "ID": pid,
                     "Distance": round(float(score), 4), "Match?": is_match})
    return images, pd.DataFrame(rows).to_markdown(index=False)

In [4]:
# Load cache pkl
with open(f"{PKL_ROOT}/gallery_cache.pkl", "rb") as f:
    gallery_cache = pickle.load(f)
print("✅ gallery_cache loaded!")

with open(f"{PKL_ROOT}/all_model_results.pkl", "rb") as f:
    all_model_results = pickle.load(f)
print("✅ all_model_results loaded!")

# Build & load semua model
models_dict = {}
for cfg in MODEL_CONFIGS:
    model = torchreid.models.build_model(
        name='osnet_x1_0',
        num_classes=cfg["num_classes"],
        pretrained=cfg["weight_url"] is None
    )
    if cfg["weight_url"] is not None:
        if not os.path.exists(cfg["weight_file"]):
            print(f"Downloading {cfg['weight_file']}...")
            gdown.download(cfg["weight_url"], cfg["weight_file"], quiet=False)
        utils.load_pretrained_weights(model, cfg["weight_file"])
        print(f"✅ Weight loaded: {cfg['weight_file']}")
    else:
        print("✅ Pretrained ImageNet — weight ImageNet digunakan")
    models_dict[cfg["name"]] = model.to(device).eval()

print("\nSemua model siap!")

✅ gallery_cache loaded!
✅ all_model_results loaded!


Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 105MB/s] 


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
✅ Pretrained ImageNet — weight ImageNet digunakan


Downloading...
From: https://drive.google.com/uc?id=1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA
To: /kaggle/working/osnet_x1_0_market1501.pth
100%|██████████| 10.4M/10.4M [00:00<00:00, 18.6MB/s]


Successfully loaded pretrained weights from "osnet_x1_0_market1501.pth"
✅ Weight loaded: osnet_x1_0_market1501.pth


Downloading...
From: https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M
To: /kaggle/working/osnet_x1_0_msmt17.pth
100%|██████████| 11.0M/11.0M [00:00<00:00, 136MB/s]

Successfully loaded pretrained weights from "osnet_x1_0_msmt17.pth"
✅ Weight loaded: osnet_x1_0_msmt17.pth

Semua model siap!


In [5]:
def build_eval_table_md():
    lines = []
    for model_name in MODEL_NAME_LIST:
        lines.append(f"### {model_name}")
        lines.append("| Subset | Metode | Rank-1 | Rank-5 | Rank-10 | mAP |")
        lines.append("|--------|--------|--------|--------|---------|-----|")
        for subset in SUBSET_LIST:
            b  = all_model_results[model_name][subset]["baseline"]
            rr = all_model_results[model_name][subset]["reranking"]
            lines.append(f"| {subset} | Baseline   | {b['r1']:.2f}% | {b['r5']:.2f}% | {b['r10']:.2f}% | {b['mAP']:.2f}% |")
            lines.append(f"|  | Re-ranking | {rr['r1']:.2f}% | {rr['r5']:.2f}% | {rr['r10']:.2f}% | {rr['mAP']:.2f}% |")
            lines.append(f"|  | **Selisih** | {rr['r1']-b['r1']:+.2f}% | {rr['r5']-b['r5']:+.2f}% | {rr['r10']-b['r10']:+.2f}% | {rr['mAP']-b['mAP']:+.2f}% |")
        lines.append("")
    return "\n".join(lines)

EVAL_TABLE_MD = build_eval_table_md()

In [6]:
def reid_demo(query_image_path, model_name, subset):
    if query_image_path is None:
        return [], [], "", "", ""

    # Validasi format nama file
    fname = os.path.basename(query_image_path)
    parts = fname.split("_")
    if not (fname.endswith(".jpg") or fname.endswith(".jpeg")) or len(parts) < 3:
        err = "⚠️ Format file salah. Gunakan format: `XXXX_cY_fZZZZ.jpg atau jpeg`"
        return [], [], err, err, err

    query_pid = parse_person_id(query_image_path)
    if query_pid is None:
        err = "⚠️ ID tidak bisa dibaca dari nama file."
        return [], [], err, err, err

    model                       = models_dict[model_name]
    cache                       = gallery_cache[model_name][subset]
    g_feats, g_labels, g_paths  = cache["feats"], cache["labels"], cache["paths"]

    # Ekstrak fitur query
    img_t = transform(Image.open(query_image_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        q_feat = model(img_t).cpu().numpy()

    # Baseline: cosine distance
    cos_dist = compute_cosine_distance(q_feat, g_feats).squeeze(0)
    base_idx = np.argsort(cos_dist)[:TOP_K]

    # Re-ranking
    rr_dist = utils.re_ranking(
        compute_cosine_distance(q_feat, g_feats),
        compute_cosine_distance(q_feat, q_feat),
        compute_cosine_distance(g_feats, g_feats),
        k1=20, k2=6, lambda_value=0.3
    ).squeeze(0)
    rr_idx = np.argsort(rr_dist)[:TOP_K]

    base_imgs, base_md = make_retrieval_outputs(base_idx, cos_dist, g_paths, g_labels, query_pid)
    rr_imgs,   rr_md   = make_retrieval_outputs(rr_idx,   rr_dist,  g_paths, g_labels, query_pid)

    query_info = f"**ID Gambar Query: {query_pid}**"
    return base_imgs, rr_imgs, base_md, rr_md, query_info


def show_query_id(filepath):
    if filepath is None:
        return ""
    fname  = os.path.basename(filepath)
    parts  = fname.split("_")
    if not (fname.endswith(".jpg") or fname.endswith(".jpeg")) or len(parts) < 3:
        return " Format file salah. Gunakan format: `XXXX_cY_fZZZZ.jpg atau jpeg`"
    pid = parse_person_id(filepath)
    return f"**ID Gambar Query: {pid}**" if pid else " ID tidak bisa dibaca dari nama file."


# ── Gradio UI ──────────────────────────────────────────────
with gr.Blocks(title="Person Re-ID Demo") as demo:
    gr.Markdown("# 🔍 Person Re-Identification Demo")
    gr.Markdown("**Dataset:** WB/WoB-ReID &nbsp;|&nbsp; **Model:** OSNet x1.0")

    with gr.Accordion(" Lihat Hasil Evaluasi Lengkap (Semua Model & Subset)", open=False):
        gr.Markdown(EVAL_TABLE_MD)

    gr.Markdown("---")
    gr.Markdown("## Upload Gambar Query")
    gr.Markdown("Upload gambar dari folder `query/` dataset. Nama file harus berformat `XXXX_cY_fZZZZ.jpg atau jpeg`.")

    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Image(
                type="filepath",
                label="Upload Gambar Query",
                sources=["upload"]          # ← hapus webcam
            )
            query_id_md = gr.Markdown("")
            model_dd    = gr.Dropdown(choices=MODEL_NAME_LIST,
                                      value="Pretrained Market-1501",
                                      label="Pilih Model")
            subset_dd   = gr.Dropdown(choices=SUBSET_LIST,
                                      value="with_bag",
                                      label="Pilih Subset Gallery")
            run_btn     = gr.Button(" Cari", variant="primary")

    gr.Markdown("---")
    gr.Markdown("## Hasil Retrieval")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📋 Baseline (Cosine Distance)")
            base_gallery = gr.Gallery(label="Top-10 Baseline",
                                      columns=5, rows=2, height=400, show_label=False)
            gr.Markdown("#### Tabel Baseline")
            base_table = gr.Markdown("")
        with gr.Column():
            gr.Markdown("### ✨ Re-ranking (k-Reciprocal)")
            rr_gallery = gr.Gallery(label="Top-10 Re-ranking",
                                    columns=5, rows=2, height=400, show_label=False)
            gr.Markdown("#### Tabel Re-ranking")
            rr_table = gr.Markdown("")

    query_input.change(fn=show_query_id, inputs=query_input, outputs=query_id_md)
    run_btn.click(fn=reid_demo,
                  inputs=[query_input, model_dd, subset_dd],
                  outputs=[base_gallery, rr_gallery, base_table, rr_table, query_id_md])

    gr.Markdown("---\n **Tips:** ✅ True = ID hasil retrieval sama dengan ID query. ❌ False = beda orang.")

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://57f0b9db0c3f8609b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
